In [1]:
# Import after modules are available
import global_snowmelt_runoff_onset.processing as processing
from global_snowmelt_runoff_onset.config import Config, Tile
import time
import logging
import dask
import gc
import psutil
import sys
from pathlib import Path
import odc.stac
import xarray as xr

In [2]:
from dask.distributed import Client
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 15.48 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34699,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:45971,Total threads: 4
Dashboard: http://127.0.0.1:40665/status,Memory: 3.87 GiB
Nanny: tcp://127.0.0.1:37549,


In [3]:
def dask_or_computed(variable):
    """
    Check if a variable is dask-backed and return status.
    
    Args:
        variable: Any variable (DataArray, Dataset, etc.)
        
    Returns:
        str: "DASK" or "COMPUTED"
    """
    # Check if it's dask-backed
    if hasattr(variable, 'data'):
        # Single DataArray
        is_dask = isinstance(variable.data, dask.array.Array)
        datatype = variable.data.dtype
        memory_gb = variable.nbytes * 1e-9

    elif hasattr(variable, 'data_vars'):
        # Dataset - check if any data variables are dask-backed
        is_dask = any([isinstance(variable[var].data, dask.array.Array)
                      for var in variable.data_vars])
        datatype = variable[list(variable.data_vars)[0]].data.dtype if variable.data_vars else None
        memory_gb = sum(var.nbytes for var in variable.data_vars.values()) * 1e-9 if variable.data_vars else 0
    else:
        # Not a dask-compatible object
        is_dask = False
        datatype = type(variable)
        memory_gb = sys.getsizeof(variable) * 1e-9  # Convert bytes to GB

    return (f"[DASK: {memory_gb:.3f}GB, dtype: {datatype}]" if is_dask
            else f"[COMPUTED: {memory_gb:.3f}GB, dtype: {datatype}]")


def monitor_memory_and_cleanup():
    """Monitor memory usage and trigger cleanup if needed."""
    # Get current memory usage
    memory_percent = psutil.virtual_memory().percent
    print(f"Current memory usage: {memory_percent:.1f}%")
    
    if memory_percent > 85:
        print(f"High memory usage: {memory_percent:.1f}%. Running cleanup...")
        gc.collect()
        
        # Force garbage collection for specific types
        for obj in gc.get_objects():
            if hasattr(obj, 'close') and callable(obj.close):
                try:
                    obj.close()
                except:
                    pass
        
        memory_after = psutil.virtual_memory().percent
        print(f"Memory usage after cleanup: {memory_after:.1f}%")

    return memory_percent

def setup_logging(tile_row: int, tile_col: int) -> None:
    """Set up logging for the tile processing."""
    log_dir = Path("logs")
    log_dir.mkdir(exist_ok=True)
    
    log_file = log_dir / f"tile_{tile_row}_{tile_col}.log"
    
    # Configure root logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler(sys.stdout)
        ]
    )
    
    # Suppress verbose logging from cloud storage and HTTP libraries
    verbose_loggers = [
        'azure.storage.blob',
        'azure.core.pipeline.policies.http_logging_policy',
        'azure.storage.blob._base_client',
        'azure.storage.blob._blob_client',
        'azure.storage.blob._container_client',
        'azure.storage.blob._download',
        'azure.storage.blob._upload_helpers',
        'azure.identity',
        'urllib3.connectionpool',
        'urllib3.util.retry',
        'requests.packages.urllib3.connectionpool',
        's3fs',
        'fsspec',
        'aiohttp.access',
    ]
    
    for logger_name in verbose_loggers:
        logging.getLogger(logger_name).setLevel(logging.WARNING)
    
    # Keep important zarr and xarray logs but reduce verbosity
    logging.getLogger('zarr').setLevel(logging.WARNING)
    logging.getLogger('xarray').setLevel(logging.INFO)
    
    logging.info(f"Logging configured for tile ({tile_row}, {tile_col})")
    logging.info("Suppressed verbose cloud storage HTTP request/response logging")


def setup_modules():
    """Set up the required processing modules (now included in repository)."""
    # The modules are now included directly in the repository
    # so we just need to verify they exist
    module_files = [
        "global_snowmelt_runoff_onset/__init__.py",
        "global_snowmelt_runoff_onset/config.py", 
        "global_snowmelt_runoff_onset/processing.py"
    ]
    
    for file_path in module_files:
        if not Path(file_path).exists():
            raise FileNotFoundError(f"Required module file not found: {file_path}")
    
    logging.info("Processing modules are available")

In [4]:
config = Config('config/global_config_v9.txt')

Configuration loaded:
config_name = global_config_v9
version = v9
resolution = 0.00072000072000072
bands = vv
mountain_snow_only = False
spatial_chunk_dim_s1_read = 2048
spatial_chunk_dim_s1_process = 512
spatial_chunk_dim_zarr_output = 2048
bbox_left = -179.999
bbox_right = 179.999
bbox_top = 81.099
bbox_bottom = -59.999
wy_start = 2015
wy_end = 2024
low_backscatter_threshold = 0.001
min_monthly_acquisitions = 1
max_allowed_days_gap_per_orbit = 30
min_years_for_median_std = 3
extend_search_window_beyond_sdd_days = 16
min_consec_snow_days_for_seasonal_snow = 56
valid_tiles_geojson_path = processing/tile_data/global_tiles_with_seasonal_snow.geojson
tile_results_path = processing/tile_data/tile_results_v9.csv
global_runoff_zarr_store_azure_path = snowmelt/snowmelt_runoff_onset/global_v9.zarr
seasonal_snow_mask_zarr_store_azure_path = snowmelt/snow_cover/global_modis_snow_cover.zarr
seasonal_snow_mask_reproject_method = rasterio


In [5]:
tiles = config.get_list_of_tiles(which='unprocessed_and_failed_skip_empty_tiles')
len(tiles)

25

In [7]:
tile_row, tile_col = 23,39
tile_row, tile_col = 23,127
tile_row, tile_col = 10,109

setup_logging(tile_row, tile_col)

In [ ]:
# try:
#     setup_modules()
# except Exception as e:
#     logging.error(f"Failed to set up required modules: {e}")
#     sys.exit(1)

In [8]:
start_time = time.time()

# Get the specific tile
tile = config.get_tile(tile_row, tile_col)
tile.start_time = start_time

In [9]:
print(f"Processing tile ({tile_row}, {tile_col})")
monitor_memory_and_cleanup()

Processing tile (10, 109)
Current memory usage: 62.7%


62.7

In [53]:
# Configure ODC for cloud access
odc.stac.configure_rio(cloud_defaults=True)

In [54]:
# Get Sentinel-1 data
print("Retrieving Sentinel-1 data...")
s1_rtc_ds = processing.get_sentinel1_rtc(
    geobox=tile.geobox,
    bands=config.bands,
    start_date=config.start_date,
    end_date=config.end_date,
    chunks_read=config.chunks_s1_read,
    fail_on_error=True,
)
# s1_rtc_ds['vv'] = s1_rtc_ds['vv'].chunk(config.chunks_s1_process) we don't do this with the serverless approach
# s1_rtc_ds['vv'] = s1_rtc_ds['vv'].astype(np.float16)
s1_rtc_ds

Retrieving Sentinel-1 data...


<xarray.Dataset> Size: 48GB
Dimensions:             (latitude: 2048, longitude: 2048, time: 2877)
Coordinates:
  * latitude            (latitude) float64 16kB 66.35 66.35 ... 64.88 64.88
  * longitude           (longitude) float64 16kB -19.27 -19.27 ... -17.8 -17.8
    spatial_ref         int32 4B 4326
  * time                (time) datetime64[ns] 23kB 2014-10-04T07:57:46.950198...
    sat:orbit_state     (time) object 23kB 'descending' ... 'ascending'
    sat:relative_orbit  (time) int16 6kB 157 11 111 118 155 ... 155 9 16 111 118
    water_year          (time) int64 23kB 2015 2015 2015 2015 ... 2025 2025 2025
    DOWY                (time) int64 23kB 4 6 13 13 16 ... 179 181 181 182 182
Data variables:
    vv                  (time, latitude, longitude) float32 48GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
Attributes:
    hemisphere:  northern

In [56]:
s1_rtc_ds['vv']=s1_rtc_ds['vv'].chunk({"latitude": 1024, "longitude": 1024, "time":10})
s1_rtc_ds

<xarray.Dataset> Size: 48GB
Dimensions:             (latitude: 2048, longitude: 2048, time: 2877)
Coordinates:
  * latitude            (latitude) float64 16kB 66.35 66.35 ... 64.88 64.88
  * longitude           (longitude) float64 16kB -19.27 -19.27 ... -17.8 -17.8
    spatial_ref         int32 4B 4326
  * time                (time) datetime64[ns] 23kB 2014-10-04T07:57:46.950198...
    sat:orbit_state     (time) object 23kB 'descending' ... 'ascending'
    sat:relative_orbit  (time) int16 6kB 157 11 111 118 155 ... 155 9 16 111 118
    water_year          (time) int64 23kB 2015 2015 2015 2015 ... 2025 2025 2025
    DOWY                (time) int64 23kB 4 6 13 13 16 ... 179 181 181 182 182
Data variables:
    vv                  (time, latitude, longitude) float32 48GB dask.array<chunksize=(10, 1024, 1024), meta=np.ndarray>
Attributes:
    hemisphere:  northern

In [59]:
config.chunks_s1_read

{'x': 2048, 'y': 2048, 'time': 1}

In [1]:
s1_rtc_ds['vv']=s1_rtc_ds['vv'].chunk({"latitude": 512, "longitude": 512, "time":50}) #xr.groupers.TimeResampler('YS-OCT')
s1_rtc_ds

NameError: name 's1_rtc_ds' is not defined

In [27]:
s1_rtc_ds.chunksizes

Frozen({'time': (157, 145, 302, 378, 388, 392, 377, 251, 209, 181, 97), 'latitude': (512, 512, 512, 512), 'longitude': (512, 512, 512, 512)})

In [28]:
s1_rtc_ds.chunks

Frozen({'time': (157, 145, 302, 378, 388, 392, 377, 251, 209, 181, 97), 'latitude': (512, 512, 512, 512), 'longitude': (512, 512, 512, 512)})

In [55]:
s1_rtc_ds['vv'].data.dask

HighLevelGraph with 4 layers.
<dask.highlevelgraph.HighLevelGraph object at 0x7f52d7d8f4d0>
 0. cfg-vv-af7f71536d1bfc3646aba944942cf1db
 1. open-vv-af7f71536d1bfc3646aba944942cf1db
 2. vv-af7f71536d1bfc3646aba944942cf1db
 3. getitem-6db256244211b0561d5b0ec44366da67

In [34]:
s1_rtc_ds.chunks.items()

ItemsView(Frozen({'time': (157, 145, 302, 378, 388, 392, 377, 251, 209, 181, 97), 'latitude': (512, 512, 512, 512), 'longitude': (512, 512, 512, 512)}))

In [ ]:
# Check if lazily loaded
print(f"Retrieved Sentinel-1 RTC dataset (s1_rtc_ds) - {dask_or_computed(s1_rtc_ds)}")
monitor_memory_and_cleanup()
tile.s1_rtc_ds_dims = dict(s1_rtc_ds.sizes)
print(f"Sentinel-1 RTC dataset dimensions: {tile.s1_rtc_ds_dims}")


In [ ]:
# Get spatiotemporal snow cover mask
print("Getting spatiotemporal snow cover mask...")
spatiotemporal_snow_cover_mask_ds = processing.get_spatiotemporal_snow_cover_mask(
    ds=s1_rtc_ds,
    bbox_gdf=tile.bbox_gdf,
    snow_phenology_store=config.snow_phenology_store,
    extend_search_window_beyond_SDD_days=config.extend_search_window_beyond_SDD_days,
    min_consec_snow_days_for_seasonal_snow=config.min_consec_snow_days_for_seasonal_snow,
).persist()
# Check if lazily loaded (should be computed/eager after .compute())
print(f"Retrieved spatiotemporal snow cover mask dataset "
                f"(spatiotemporal_snow_cover_mask_ds) - {dask_or_computed(spatiotemporal_snow_cover_mask_ds)}")
monitor_memory_and_cleanup()
spatiotemporal_snow_cover_mask_ds

In [ ]:
# Get mountain inventory if needed
if config.mountain_snow_only:
    gmba_clipped_gdf = processing.get_gmba_mountain_inventory(tile.bbox_gdf)
else:
    gmba_clipped_gdf = None

# Apply masks
print("Applying masks...")
s1_rtc_masked_ds = processing.apply_all_masks(
    s1_rtc_ds=s1_rtc_ds,#.chunk(config.chunks_s1_process), # .chunk(config.chunks_s1_process)
    gmba_clipped_gdf=gmba_clipped_gdf,
    spatiotemporal_snow_cover_mask_ds=spatiotemporal_snow_cover_mask_ds.chunk({"latitude": 1024, "longitude": 1024, "water_year": 1}),#.chunk(chunks='auto'),#.chunk({"latitude": 64, "longitude": 64, "water_year": 1}),#.chunk({"latitude": 512, "longitude": 512, "water_year": 1}),
    water_years=config.water_years,
)
# Check if lazily loaded
print(f"Applied all masks to S1 RTC dataset "
                f"(s1_rtc_masked_ds) - {dask_or_computed(s1_rtc_masked_ds)}")
monitor_memory_and_cleanup()

s1_rtc_masked_ds

In [ ]:
s1_rtc_masked_ds = processing.remove_bad_scenes_and_border_noise(
    s1_rtc_masked_ds, config.low_backscatter_threshold
)

print(f"Removed bad scenes and border noise from S1 RTC "
                f"dataset (s1_rtc_masked_ds) - {dask_or_computed(s1_rtc_masked_ds)}")
monitor_memory_and_cleanup()

s1_rtc_masked_ds

In [ ]:
s1_rtc_masked_filtered_ds = s1_rtc_masked_ds.groupby("water_year").map(
    lambda group: processing.filter_insufficient_pixels_per_orbit(
        s1_rtc_masked_ds=group,
        spatiotemporal_snow_cover_mask_ds=spatiotemporal_snow_cover_mask_ds,
        min_monthly_acquisitions=config.min_monthly_acquisitions,
        max_allowed_days_gap_per_orbit=config.max_allowed_days_gap_per_orbit,
    )
)
# Check if lazily loaded
print(f"Filtered S1 RTC dataset by acquisitions and gaps "
                f"(s1_rtc_masked_filtered_ds) - {dask_or_computed(s1_rtc_masked_filtered_ds)}")
monitor_memory_and_cleanup()

s1_rtc_masked_filtered_ds

In [ ]:
temporal_resolution_da = processing.get_temporal_resolution(
    s1_rtc_masked_filtered_ds, spatiotemporal_snow_cover_mask_ds
)
# Check if lazily loaded
print(f"Calculated temporal resolution data array "
                f"(temporal_resolution_da) - {dask_or_computed(temporal_resolution_da)}")
temporal_resolution_da

In [ ]:
runoff_onsets_da = s1_rtc_masked_filtered_ds.groupby("water_year").apply(
    processing.calculate_runoff_onset,
    returned_dates_format="dowy",
    return_constituent_runoff_onsets=False,
)
# Check if lazily loaded
print(f"Calculated runoff onsets data array "
                f"(runoff_onsets_da) - {dask_or_computed(runoff_onsets_da)}")
monitor_memory_and_cleanup()

tile.runoff_onsets_dims = dict(runoff_onsets_da.sizes)

runoff_onsets_da

In [ ]:
median_da, mad_da = processing.median_and_mad_with_min_obs(
    da=runoff_onsets_da,
    dim="water_year",
    min_count=config.min_years_for_median_std
)
# Check if lazily loaded
print(f"Calculated median data array (median_da) - {dask_or_computed(median_da)}")
print(f"Calculated MAD data array (mad_da) - {dask_or_computed(mad_da)}")
monitor_memory_and_cleanup()

median_da

In [ ]:
median_temporal_resolution_da = processing.median_with_min_obs(
    da=temporal_resolution_da,
    dim="water_year",
    min_count=config.min_years_for_median_std
)
# Check if lazily loaded
print(f"Calculated median temporal resolution data array "
      f"(median_temporal_resolution_da) - {dask_or_computed(median_temporal_resolution_da)}")
monitor_memory_and_cleanup()

median_temporal_resolution_da

In [ ]:
runoff_onsets_ds = processing.dataarrays_to_dataset(
    runoff_onsets_da=runoff_onsets_da,
    median_da=median_da,
    mad_da=mad_da,
    water_years=config.water_years,
    temporal_resolution_da=temporal_resolution_da,
    median_temporal_resolution_da=median_temporal_resolution_da,
)
# Check if dataset is lazy (handle chunking inconsistencies)
try:
    status = dask_or_computed(runoff_onsets_ds)
except ValueError as e:
    if "inconsistent chunks" in str(e):
        print(f"Dataset has inconsistent chunks: {e}")
        # Fix chunking inconsistencies
        runoff_onsets_ds = runoff_onsets_ds.unify_chunks()
        print("Applied unify_chunks() to fix inconsistent chunking")
        # Try again to check lazy loading
        status = dask_or_computed(runoff_onsets_ds)
    else:
        raise

print(f"Created runoff onsets dataset (runoff_onsets_ds) - {status}")
monitor_memory_and_cleanup()

runoff_onsets_ds

In [ ]:
# Reindex to global coordinates
global_ds = xr.open_zarr(config.global_runoff_store, consolidated=True)
global_subset_ds = global_ds.sel(
    latitude=runoff_onsets_ds.latitude,
    longitude=runoff_onsets_ds.longitude,
    method="nearest",
)
runoff_onsets_reindexed_ds = runoff_onsets_ds.assign_coords(
    latitude=global_subset_ds.latitude, 
    longitude=global_subset_ds.longitude
)

print(f"Reindexed to global coordinates (runoff_onsets_reindexed_ds) - {dask_or_computed(runoff_onsets_reindexed_ds)}")
monitor_memory_and_cleanup()

runoff_onsets_reindexed_ds

In [ ]:
# from concurrent.futures import ThreadPoolExecutor
# with dask.config.set(pool=ThreadPoolExecutor(16), scheduler="threads"):
#     runoff_onsets_reindexed_ds.drop_vars("spatial_ref").chunk(
#         config.chunks_zarr_output
#     ).to_zarr(
#         config.global_runoff_store, region="auto", mode="r+", consolidated=True
#     )
# print("Results written to global zarr store")
# monitor_memory_and_cleanup()

In [ ]:
runoff_onsets_reindexed_ds.drop_vars("spatial_ref").chunk(
    config.chunks_zarr_output
).to_zarr(
    config.global_runoff_store, region="auto", mode="r+", consolidated=True
)
print("Results written to global zarr store")
monitor_memory_and_cleanup()

In [ ]:
global_ds = xr.open_zarr(config.global_runoff_store, consolidated=True)
global_subset_ds = global_ds.sel(
    latitude=runoff_onsets_ds.latitude,
    longitude=runoff_onsets_ds.longitude,
    method="nearest",
)

tile_median_temporal_resolution = global_subset_ds['temporal_resolution'].median(
    dim=["latitude", "longitude"]
).compute()
tile_pixel_count = global_subset_ds['temporal_resolution'].count(
    dim=["latitude", "longitude"]
).compute()

# Log if these are lazy
print(f"Tile median temporal resolution (tile_median_temporal_resolution) - "
        f"{dask_or_computed(tile_median_temporal_resolution)}")
print(f"Tile pixel count (tile_pixel_count) - "
        f"{dask_or_computed(tile_pixel_count)}")
monitor_memory_and_cleanup()

In [ ]:
# Store temporal resolution metrics
for water_year in config.water_years:
    if water_year in tile_median_temporal_resolution.water_year:
        temporal_resolution = tile_median_temporal_resolution.sel(
            water_year=water_year
        ).values
        setattr(tile, f"tr_{water_year}", round(float(temporal_resolution), 3))

    if water_year in tile_pixel_count.water_year:
        pixel_count = tile_pixel_count.sel(water_year=water_year).values
        setattr(tile, f"pix_ct_{water_year}", int(pixel_count))

print("Tile median temporal resolution and pixel count per water year stored in tile object")
monitor_memory_and_cleanup()